# Continuous pushing: grounding and action chunks

This standalone experiment trains a compact visual-language action policy to push the instructed object left or right. It studies compositional colour/shape grounding, four-action chunks and robustness.

Six objects are visible. The blue disk is never a commanded training target, but appears as a distractor with auxiliary visual labels. Commands follow a fixed French grammar; this is not an unrestricted language model.

Run top to bottom on CPU or a Colab GPU. The notebook generates its own data and exports measured metrics, weights and animations. The task models horizontal quasi-static contact only. See notebook 02 for a separate grid pushing benchmark with obstacles.


In [ ]:
import importlib.util, subprocess, sys
packages={'torch':'torch','numpy':'numpy','PIL':'Pillow','matplotlib':'matplotlib','IPython':'ipython'}
missing=[p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import copy, csv, json, math, random, time, zipfile
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage
SEED=17
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4,torch.get_num_threads()))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cuda': torch.cuda.manual_seed_all(SEED)
RUN=Path('mini_vla_lab_results'); RUN.mkdir(exist_ok=True)
print('Device:',DEVICE,'| PyTorch:',torch.__version__)
if DEVICE.type=='cuda': print(torch.cuda.get_device_name(0))

## 1 · The simulator and its scripted teacher

The pusher starts on the upper edge. The demonstrator first aligns its x-coordinate behind the requested object, then its y-coordinate, then pushes. Its access to exact object positions creates training labels. **The neural policy only receives the rendered image, instruction and its own position.**

Actions are normalised in `[-1,1]`. Each simulation step multiplies them by `STEP × actuator_gain`. Horizontal contact moves objects; vertical contact and inertia are not modelled. This simplification makes the experiment accessible, but rules out claims of realistic physical control.

Success requires reaching the requested edge with the right object while preserving distractors. A separate failure reason distinguishes wrong-object movement from timeout.

In [ ]:
SIZE, GRID, STEP, GAP, HORIZON = 64, 16, 0.035, 0.075, 4
MAX_STEPS=100
COLORS=['rouge','vert','bleu']; SHAPES=['carre','disque']
PALETTE=[(235,65,75),(45,205,110),(60,125,245)]
OBJECT_COLORS=np.repeat(np.arange(3),2)
OBJECT_SHAPES=np.tile(np.arange(2),3)
VOCAB={w:i for i,w in enumerate('pousse le carre disque rouge vert bleu vers la gauche droite'.split())}
ALL_TASKS=[(c,s,d) for c in range(3) for s in range(2) for d in [-1,1]]
TRAIN_TASKS=[t for t in ALL_TASKS if t[:2]!=(2,1)]
HELDOUT_TASKS=[t for t in ALL_TASKS if t[:2]==(2,1)]

def instruction(color,shape,direction):
    return f"pousse le {SHAPES[shape]} {COLORS[color]} vers la {'droite' if direction==1 else 'gauche'}"

def tokenize(text):
    w=text.lower().strip().split()
    if len(w)!=7 or w[:2]!=['pousse','le'] or w[4:6]!=['vers','la'] or w[2] not in SHAPES or w[3] not in COLORS or w[6] not in ['gauche','droite']:
        raise ValueError('Use: pousse le carre/disque rouge/vert/bleu vers la gauche/droite')
    return np.array([VOCAB[x] for x in w],dtype=np.int64)

class World:
    def __init__(self,seed,task):
        self.rng=np.random.default_rng(seed)
        self.color,self.shape,self.direction=task
        self.target=2*self.color+self.shape
        ys=self.rng.permutation(np.linspace(.19,.83,6))+self.rng.uniform(-.008,.008,6)
        self.objects=np.column_stack([self.rng.uniform(.33,.67,6),ys]).astype(np.float32)
        self.initial=self.objects.copy()
        # Important: this draw has no dependence on task/direction.
        self.pusher=np.array([self.rng.uniform(.12,.88),.055],dtype=np.float32)
        self.steps=0

    @property
    def text(self): return instruction(self.color,self.shape,self.direction)

    def distractor_moved(self):
        ids=[i for i in range(6) if i!=self.target]
        return bool(np.abs(self.objects[ids]-self.initial[ids]).max()>.04)

    def success(self):
        x=self.objects[self.target,0]
        arrived=x>=.85 if self.direction==1 else x<=.15
        return bool(arrived and not self.distractor_moved())

    def render(self):
        im=Image.new('RGB',(SIZE,SIZE),(18,25,40)); dr=ImageDraw.Draw(im)
        dr.rectangle((0,0,round(.15*(SIZE-1)),SIZE),fill=(31,47,63))
        dr.rectangle((round(.85*(SIZE-1)),0,SIZE,SIZE),fill=(31,47,63))
        for i,(x,y) in enumerate(self.objects*(SIZE-1)):
            box=(x-3,y-3,x+3,y+3); rgb=PALETTE[OBJECT_COLORS[i]]
            if OBJECT_SHAPES[i]==0: dr.rectangle(box,fill=rgb)
            else: dr.ellipse(box,fill=rgb)
        x,y=self.pusher*(SIZE-1)
        dr.ellipse((x-1.8,y-1.8,x+1.8,y+1.8),fill=(245,248,255))
        return np.asarray(im).copy()

    def step(self,action,gain=1.):
        old=self.pusher.copy()
        new=np.clip(old+STEP*gain*np.clip(action,-1,1),.03,.97)
        for obj in self.objects:
            if abs(new[1]-obj[1])<.043:
                if new[0]>old[0] and old[0]<=obj[0]-GAP+.008 and new[0]>obj[0]-GAP:
                    obj[0]=np.clip(new[0]+GAP,.055,.945)
                elif new[0]<old[0] and old[0]>=obj[0]+GAP-.008 and new[0]<obj[0]+GAP:
                    obj[0]=np.clip(new[0]-GAP,.055,.945)
        self.pusher=new.astype(np.float32); self.steps+=1

def expert(w):
    xy=w.objects[w.target]; dy=xy[1]-w.pusher[1]
    behind=w.direction*(w.pusher[0]-xy[0])<-.055
    if abs(dy)<.022 and behind:
        return np.array([w.direction,np.clip(14*dy,-1,1)],np.float32)
    dx=(xy[0]-w.direction*.105)-w.pusher[0]
    if abs(dx)>.012:
        return np.array([np.clip(14*dx,-1,1),0],np.float32)
    return np.array([0,np.clip(14*dy,-1,1)],np.float32)

def teacher_chunk(w):
    clone=copy.deepcopy(w); actions=[]
    for _ in range(HORIZON):
        a=expert(clone); actions.append(a); clone.step(a)
    return np.stack(actions)

# The same seed means exactly the same initial scene, for opposite instructions.
a,b=World(42,(0,0,-1)),World(42,(0,0,1))
assert np.array_equal(a.render(),b.render()) and np.array_equal(a.pusher,b.pusher)
assert len(TRAIN_TASKS)==10 and len(HELDOUT_TASKS)==2
print('10 training tasks; 2 held-out tasks:',[instruction(*t) for t in HELDOUT_TASKS])
fig,ax=plt.subplots(figsize=(4,4)); ax.imshow(a.render()); ax.set_title('Same image; direction must come from language'); ax.axis('off'); plt.show()

## 2 · Generate demonstrations and auxiliary labels

The dataset stores each observation and **four future expert actions**. We also supervise visual grounding using known simulation positions: a colour attention map should cover both objects of that colour, a shape map should cover all objects of that shape, and their combination should locate the requested object.

This is strong attribute supervision. In particular, the blue disk is visible in training and participates in colour/shape labels, although it is never the target of a training action sequence. Our held-out test therefore asks whether learned attributes can be **recombined into an unseen commanded task**, not whether the model can recognise a completely new object.

The split is by complete episode, with separate seed ranges. The validation set only contains the ten training task types. All final test tasks, seeds and stress settings are declared before training.

In [ ]:
TRAIN_EPISODES=480
VAL_EPISODES=60
TRAIN_SEEDS=list(range(30000,30000+TRAIN_EPISODES))
VAL_SEEDS=list(range(40000,40000+VAL_EPISODES))
TEST_SEEDS=list(range(50000,50060))
HELDOUT_SEEDS=list(range(60000,60024))
STRESS_SEEDS=list(range(70000,70030))
assert len(set(TRAIN_SEEDS+VAL_SEEDS+TEST_SEEDS+HELDOUT_SEEDS+STRESS_SEEDS))==sum(map(len,[TRAIN_SEEDS,VAL_SEEDS,TEST_SEEDS,HELDOUT_SEEDS,STRESS_SEEDS]))

def collect(seeds,perturb=False):
    records=[]; done=[]
    for i,seed in enumerate(seeds):
        task=TRAIN_TASKS[i%len(TRAIN_TASKS)]; assert task not in HELDOUT_TASKS
        w=World(seed,task)
        for t in range(MAX_STEPS):
            a=expert(w)
            if t%2==0:
                records.append((w.render(),tokenize(w.text),w.pusher.copy(),teacher_chunk(w),
                                w.objects.copy(),w.target))
            w.step(a+(w.rng.normal(0,.055,2) if perturb else 0))
            if w.success(): break
        done.append(w.success())
    images,text,state,actions,positions,ids=map(np.stack,zip(*records))
    print(len(seeds),'episodes;',len(records),'observations; expert success',f'{np.mean(done):.1%}')
    return {'image':torch.from_numpy(images).permute(0,3,1,2),'text':torch.from_numpy(text),
            'state':torch.from_numpy(state),'action':torch.from_numpy(actions),
            'positions':torch.from_numpy(positions),'target_id':torch.from_numpy(ids).long()}

start=time.perf_counter(); train=collect(TRAIN_SEEDS,True); val=collect(VAL_SEEDS)
print('Generation seconds:',round(time.perf_counter()-start,1))

## 3 · Model: factorised grounding and action chunks

**Vision:** a small CNN turns the image into 256 spatial feature vectors. **Language:** learned word embeddings encode the fixed seven-word grammar. Colour and shape word positions are specified explicitly; this is a structured-language controller.

Two learned compatibility scores are computed for each visual patch:

`colour_score = Q_colour · K_colour / √32`

`shape_score = Q_shape · K_shape / √32`

Their sum is passed through a softmax. Adding log-scores is equivalent to multiplying the two unnormalised compatibilities, encouraging their intersection. The resulting weights aggregate the visual features and patch coordinates. No handcrafted colour threshold or shape detector is used.

The action MLP combines that context, the instruction, the pusher position and its offset from the estimated target. It outputs **4 × 2 action values**. The model does not plan a full trajectory: it predicts a short chunk and replans after observing again.

`loss = action MSE + 4 × location MSE + 0.08 × (colour CE + shape CE + joint CE)`

All ground-truth object coordinates are used only for losses, the expert and metrics. At inference, the model obtains coordinates only through its learned visual attention. This is supervised imitation learning, not RL.

In [ ]:
class MiniVLA(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision=nn.Sequential(nn.Conv2d(3,16,3,stride=2,padding=1),nn.ReLU(),
                                 nn.Conv2d(16,24,3,padding=1),nn.ReLU(),nn.AvgPool2d(2))
        self.embedding=nn.Embedding(len(VOCAB),16)
        self.language=nn.Sequential(nn.Linear(112,48),nn.ReLU(),nn.Linear(48,32))
        self.color_query=nn.Linear(16,32); self.shape_query=nn.Linear(16,32)
        self.color_key=nn.Linear(24,32); self.shape_key=nn.Linear(24,32)
        axis=(torch.arange(GRID)*4+1)/(SIZE-1)
        y,x=torch.meshgrid(axis,axis,indexing='ij')
        self.register_buffer('coords',torch.stack([x,y],-1).reshape(1,GRID*GRID,2))
        self.head=nn.Sequential(nn.Linear(24+32+4,128),nn.ReLU(),nn.Linear(128,96),
                                nn.ReLU(),nn.Linear(96,HORIZON*2),nn.Tanh())

    def forward(self,image,text,state):
        features=self.vision(image).flatten(2).transpose(1,2)
        words=self.embedding(text)
        color_logits=(self.color_key(features)*self.color_query(words[:,3]).unsqueeze(1)).sum(-1)/math.sqrt(32)
        shape_logits=(self.shape_key(features)*self.shape_query(words[:,2]).unsqueeze(1)).sum(-1)/math.sqrt(32)
        joint=color_logits+shape_logits; attention=joint.softmax(-1)
        location=(attention.unsqueeze(-1)*self.coords).sum(1)
        context=(attention.unsqueeze(-1)*features).sum(1)
        language=self.language(words.flatten(1))
        actions=self.head(torch.cat([context,language,location-state,state],-1)).reshape(-1,HORIZON,2)
        return actions,location,(color_logits,shape_logits,joint)

model=MiniVLA().to(DEVICE)
PARAMETERS=sum(p.numel() for p in model.parameters())
print(f'{PARAMETERS:,} parameters, {PARAMETERS*4/1024:.1f} KiB of float32 weights')

def batch(data,idx,augment=False):
    b={k:v[idx].to(DEVICE) for k,v in data.items()}
    im=b['image'].float()/255
    if augment:
        # Moderate randomisation; the final dark/noisy tests are deliberately stronger.
        brightness=torch.empty(len(im),1,1,1,device=DEVICE).uniform_(.65,1.2)
        channels=torch.empty(len(im),3,1,1,device=DEVICE).uniform_(.9,1.1)
        im=(im*brightness*channels+torch.randn_like(im)*.012).clamp(0,1)
    b['image']=im
    return b

def grounding_targets(b):
    dist=(model.coords.unsqueeze(1)-b['positions'].unsqueeze(2)).square().sum(-1)
    maps=torch.exp(-dist/(2*.035**2))
    ids=b['target_id']; rows=torch.arange(len(ids),device=DEVICE)
    colors=torch.tensor(OBJECT_COLORS,device=DEVICE)
    shapes=torch.tensor(OBJECT_SHAPES,device=DEVICE)
    cm=(maps*(colors.unsqueeze(0)==colors[ids,None]).unsqueeze(-1)).sum(1)
    sm=(maps*(shapes.unsqueeze(0)==shapes[ids,None]).unsqueeze(-1)).sum(1)
    jm=maps[rows,ids]
    return [m/m.sum(-1,keepdim=True).clamp_min(1e-9) for m in [cm,sm,jm]]

@torch.inference_mode()
def validation_metrics():
    model.eval(); total=action_error=location_error=0.
    for i in range(0,len(val['image']),256):
        b=batch(val,slice(i,i+256)); a,xy,_=model(b['image'],b['text'],b['state'])
        n=len(a); target=b['positions'][torch.arange(n,device=DEVICE),b['target_id']]
        action_error+=F.mse_loss(a,b['action']).item()*n
        location_error+=(xy-target).norm(dim=1).mean().item()*n; total+=n
    return {'action_mse':action_error/total,'location_error':location_error/total}

baseline=validation_metrics(); print('Untrained validation:',baseline)

## 4 · Train and select a checkpoint using validation only

There are two independent ideas to understand: the loss teaches the network to reproduce action sequences; the auxiliary labels help it find the requested object efficiently. Moderate camera randomisation is applied only during training.

The best checkpoint is selected by validation action MSE. Test results are not used to select it. The files are saved before the final evaluation, so the trained weights remain available if you interrupt the benchmark. For a fresh experiment, restart from the first cell.

In [ ]:
STEPS=2500
BATCH_SIZE=96
LEARNING_RATE=.0015
LOCATION_WEIGHT=4.0
ATTRIBUTE_WEIGHT=.08
CAMERA_AUGMENTATION=True
optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
history=[]; best_value=baseline['action_mse']; best_state=copy.deepcopy(model.state_dict())
started=time.perf_counter()
try:
    for step in range(1,STEPS+1):
        model.train(); idx=torch.randint(len(train['image']),(BATCH_SIZE,))
        b=batch(train,idx,augment=CAMERA_AUGMENTATION)
        actions,xy,logits=model(b['image'],b['text'],b['state'])
        target=b['positions'][torch.arange(len(idx),device=DEVICE),b['target_id']]
        imitation=F.mse_loss(actions,b['action'])
        localization=F.mse_loss(xy,target)
        maps=grounding_targets(b)
        attention_loss=sum(-(label*F.log_softmax(logit,-1)).sum(-1).mean() for logit,label in zip(logits,maps))
        loss=imitation+LOCATION_WEIGHT*localization+ATTRIBUTE_WEIGHT*attention_loss
        optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.); optimizer.step()
        if step%100==0 or step==STEPS:
            metrics=validation_metrics(); elapsed=time.perf_counter()-started
            history.append({'step':step,'train_action_mse':imitation.item(),**metrics,'seconds':elapsed})
            if metrics['action_mse']<best_value:
                best_value=metrics['action_mse']; best_state=copy.deepcopy(model.state_dict())
            print(f"{step}/{STEPS} | val actions {metrics['action_mse']:.4f} | location {metrics['location_error']:.3f} | "
                  f"elapsed {elapsed/60:.1f} min | ETA {elapsed/step*(STEPS-step)/60:.1f} min")
except KeyboardInterrupt:
    print('Interrupted: restoring the best evaluated checkpoint.')
model.load_state_dict(best_state); model.eval()
TRAIN_SECONDS=time.perf_counter()-started
after=validation_metrics()
torch.save({k:v.cpu() for k,v in model.state_dict().items()},RUN/'weights.pt')
print('Before:',baseline,'After:',after)
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
ax[0].plot([r['step'] for r in history],[r['action_mse'] for r in history]); ax[0].set(xlabel='updates',ylabel='validation action MSE')
ax[1].plot([r['step'] for r in history],[r['location_error'] for r in history]); ax[1].set(xlabel='updates',ylabel='validation position error')
fig.tight_layout(); fig.savefig(RUN/'learning_curves.png',dpi=160); plt.show()

## 5 · Deploy the learned policy inside the simulator

The deployment loop is `render → predict chunk → execute k actions → render again`. `k=1` gives frequent feedback; `k=4` makes fewer model calls but allows errors to accumulate before the next observation.

The policy interface below accepts only an image, text and pusher state. Ground truth cannot enter that function. The expert uses a separate path in the evaluator.

The pusher displacement test applies an external +0.10 vertical offset at simulation step 12. A displacement occurring inside a four-action chunk is not observed until the next prediction. The slower-actuator test changes action scaling, not the model output.

In [ ]:
@torch.inference_mode()
def policy(image,text,state):
    model.eval()
    im=torch.from_numpy(np.asarray(image).copy()).permute(2,0,1).unsqueeze(0).float().to(DEVICE)/255
    tokens=torch.tensor(tokenize(text),device=DEVICE).unsqueeze(0)
    state=torch.tensor(np.asarray(state),device=DEVICE).unsqueeze(0)
    actions,location,logits=model(im,tokens,state)
    return actions[0].cpu().numpy(),location[0].cpu().numpy(),[x[0].softmax(-1).cpu().numpy() for x in logits]

def camera(image,mode,rng):
    x=image.astype(np.float32)/255
    if mode=='dark': x*=.35
    elif mode=='noise': x+=rng.normal(0,.10,x.shape)
    elif mode=='blank': x[:]=0
    return (np.clip(x,0,1)*255).astype(np.uint8)

def rollout(seed,task,condition='clean',execute=1,teacher=False,record=False):
    w=World(seed,task); rng=np.random.default_rng(seed+900000)
    frames=[]; calls=0; interfered=False
    text=w.text
    if condition=='wrong_color': text=instruction((task[0]+1)%3,task[1],task[2])
    if condition=='wrong_shape': text=instruction(task[0],1-task[1],task[2])
    if condition=='wrong_direction': text=instruction(task[0],task[1],-task[2])
    while w.steps<MAX_STEPS:
        image=camera(w.render(),condition,rng)
        if teacher:
            actions=teacher_chunk(w); location=w.objects[w.target].copy(); maps=None
        else: actions,location,maps=policy(image,text,w.pusher)
        calls+=1
        for a in actions[:execute]:
            if w.steps>=MAX_STEPS: break
            if condition=='displacement' and w.steps==12 and not interfered:
                w.pusher[1]=np.clip(w.pusher[1]+.10,.03,.97); interfered=True
            if record:
                frames.append({'image':w.render(),'camera':image,'pusher':w.pusher.copy(),
                               'location':location.copy(),'maps':maps,'actions':actions.copy(),
                               'step':w.steps,'text':text,'success':False})
            w.step(a,gain=.6 if condition=='slow_actuator' else 1.)
            if w.success(): break
        if w.success(): break
    if record:
        frames.append({'image':w.render(),'camera':camera(w.render(),condition,rng),'pusher':w.pusher.copy(),
                       'location':location.copy(),'maps':maps,'actions':actions.copy(),
                       'step':w.steps,'text':text,'success':w.success()})
    reason='success' if w.success() else ('distractor_moved' if w.distractor_moved() else 'timeout')
    return {'seed':int(seed),'color':task[0],'shape':task[1],'direction':task[2],'condition':condition,
            'execute':execute,'teacher':teacher,'success':w.success(),'reason':reason,
            'steps':w.steps,'model_calls':0 if teacher else calls},frames

# Development check on validation scenes, before inspecting the final tests.
dev=[]
for i,seed in enumerate(VAL_SEEDS[:10]):
    result,_=rollout(seed,TRAIN_TASKS[i%len(TRAIN_TASKS)]); dev.append(result)
print('Validation rollouts:',sum(r['success'] for r in dev),'/',len(dev))

## 6 · A benchmark with explicit questions

**Composition:** can a blue disk be commanded although that target combination was excluded from demonstrations? **Instruction use:** does replacing colour, shape or direction change the outcome on identical initial scenes? **Feedback:** does executing four actions between observations affect success or model-call count? **Robustness:** what changes under camera and actuator perturbations?

All wrong-instruction runs are scored against the original task. They test sensitivity, not language understanding outside this grammar. Expert stress runs expose limitations of the simulator/controller and time budget. A fixed 100-step budget may penalise slow actuators.

Intervals are Wilson 95% intervals over episodes for this single trained model. They do not include training-seed variability. A larger study should repeat training with multiple seeds. Do not tune on these results; use fresh final seeds after any design changes.

In [ ]:
def wilson(k,n):
    z=1.96; p=k/n; den=1+z*z/n
    center=(p+z*z/(2*n))/den; half=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return [max(0.,center-half),min(1.,center+half)]

SPECS=[
    ('expert_seen',TEST_SEEDS,TRAIN_TASKS,'clean',1,True),
    ('seen_k1',TEST_SEEDS,TRAIN_TASKS,'clean',1,False),
    ('seen_k4',TEST_SEEDS,TRAIN_TASKS,'clean',4,False),
    ('heldout_blue_disk',HELDOUT_SEEDS,HELDOUT_TASKS,'clean',1,False),
    ('wrong_color',STRESS_SEEDS,TRAIN_TASKS,'wrong_color',1,False),
    ('wrong_shape',STRESS_SEEDS,TRAIN_TASKS,'wrong_shape',1,False),
    ('wrong_direction',STRESS_SEEDS,TRAIN_TASKS,'wrong_direction',1,False),
    ('blank_image',STRESS_SEEDS,TRAIN_TASKS,'blank',1,False),
    ('dark_camera',STRESS_SEEDS,TRAIN_TASKS,'dark',1,False),
    ('noisy_camera',STRESS_SEEDS,TRAIN_TASKS,'noise',1,False),
    ('slow_actuator',STRESS_SEEDS,TRAIN_TASKS,'slow_actuator',1,False),
    ('expert_slow',STRESS_SEEDS,TRAIN_TASKS,'slow_actuator',1,True),
    ('displacement_k1',STRESS_SEEDS,TRAIN_TASKS,'displacement',1,False),
    ('displacement_k4',STRESS_SEEDS,TRAIN_TASKS,'displacement',4,False),
    ('expert_displacement',STRESS_SEEDS,TRAIN_TASKS,'displacement',1,True)]
rows=[]; summary={}; eval_start=time.perf_counter()
for name,seeds,tasks,condition,execute,teacher in SPECS:
    local=[]
    for i,seed in enumerate(seeds):
        row,_=rollout(seed,tasks[i%len(tasks)],condition,execute,teacher)
        row['benchmark']=name; local.append(row)
    rows+=local; k=sum(r['success'] for r in local); n=len(local)
    summary[name]={'successes':k,'episodes':n,'success_rate':k/n,'wilson_95':wilson(k,n),
                   'mean_steps':float(np.mean([r['steps'] for r in local])),
                   'mean_model_calls':float(np.mean([r['model_calls'] for r in local]))}
    lo,hi=summary[name]['wilson_95']
    print(f'{name:23} {k:2}/{n}  {k/n:6.1%}  CI [{lo:.0%}, {hi:.0%}]  calls {summary[name]["mean_model_calls"]:.1f}')
EVAL_SECONDS=time.perf_counter()-eval_start
fig,ax=plt.subplots(figsize=(10,6))
names=list(summary); values=np.array([summary[x]['success_rate'] for x in names])
errors=np.array([[values[i]-summary[x]['wilson_95'][0],summary[x]['wilson_95'][1]-values[i]] for i,x in enumerate(names)]).T.clip(0)
ax.barh(names,values,xerr=errors,color=['#4e8195' if x.startswith('expert') else '#36b89f' for x in names],capsize=3)
ax.set(xlim=(0,1.05),xlabel='Episode success rate, with Wilson 95% intervals'); ax.invert_yaxis()
fig.tight_layout(); fig.savefig(RUN/'benchmark.png',dpi=160); plt.show()

## 7 · Inspect the behaviour, attention and predicted chunk

The annotated animation shows the RGB scene, the estimated target location and the predicted four-action sequence. The right panel visualises the joint attention. The action arrows are a kinematic projection of the model output; they are not a separately planned collision-free path. Attention stays fixed while a chunk is executed, then refreshes on the next model call.

Examples are chosen deterministically: first success and first failure when available. A lower score in a harder task is useful evidence, not something to hide.

In [ ]:
def font(size):
    for name in ['DejaVuSans.ttf','arial.ttf']:
        try: return ImageFont.truetype(name,size)
        except OSError: pass
    return ImageFont.load_default()

def annotated_gif(frames,filename,label):
    output=[]; f_title=font(18); f_body=font(14)
    for record in frames:
        canvas=Image.new('RGB',(840,480),(12,18,29)); draw=ImageDraw.Draw(canvas)
        draw.text((24,14),'MINI-VLA LAB  /  '+label,fill=(85,220,185),font=f_title)
        draw.text((24,44),record['text'],fill='white',font=f_title)
        scene=Image.fromarray(record['image']).resize((340,340),Image.Resampling.NEAREST)
        canvas.paste(scene,(24,88))
        loc=record['location']*339+np.array([24,88]); draw.ellipse((loc[0]-7,loc[1]-7,loc[0]+7,loc[1]+7),outline='yellow',width=2)
        p=record['pusher'].copy(); points=[tuple(p*339+np.array([24,88]))]
        for action in record['actions']:
            p=np.clip(p+STEP*action,0,1); points.append(tuple(p*339+np.array([24,88])))
        draw.line(points,fill=(255,215,80),width=3)
        if record['maps'] is not None:
            att=record['maps'][2].reshape(GRID,GRID)
            heat=(plt.get_cmap('magma')(att/max(float(att.max()),1e-8))[:,:,:3]*255).astype(np.uint8)
            heat=Image.fromarray(heat).resize((256,256),Image.Resampling.NEAREST)
            canvas.paste(heat,(450,110))
        draw.text((450,88),'Learned joint attention',fill='white',font=f_body)
        draw.text((394,388),'Yellow circle: estimated target',fill=(255,215,80),font=f_body)
        draw.text((394,412),'Yellow line: predicted action chunk',fill=(255,215,80),font=f_body)
        status='SUCCESS' if record['success'] else f'Step {record["step"]}'
        draw.text((24,447),status,fill=(85,220,185) if record['success'] else 'white',font=f_title)
        output.append(canvas)
    path=RUN/filename
    output[0].save(path,save_all=True,append_images=output[1:],duration=85,loop=0)
    display(DisplayImage(filename=str(path)))
    return path

for benchmark in ['seen_k1','heldout_blue_disk','displacement_k4']:
    group=[r for r in rows if r['benchmark']==benchmark]
    for success in [True,False]:
        chosen=next((r for r in group if r['success']==success),None)
        if chosen:
            task=(chosen['color'],chosen['shape'],chosen['direction'])
            _,frames=rollout(chosen['seed'],task,chosen['condition'],chosen['execute'],record=True)
            annotated_gif(frames,f'{benchmark}_{"success" if success else "failure"}.gif',benchmark)

## 8 · Counterfactual instructions, identical scene

Change the colour, shape or direction while keeping the initial pixels and state identical. These examples are qualitative, separate from the quantitative benchmark. Watch how the attended object and the route change.

You can change `DEMO_SEED` and `DEMO_TASKS` below. Valid colours are 0/1/2 (red/green/blue), shapes 0/1 (square/disk), and directions −1/+1 (left/right).

In [ ]:
DEMO_SEED=81001
DEMO_TASKS=[(0,0,-1),(0,0,1),(0,1,1),(2,1,1)]
first=World(DEMO_SEED,DEMO_TASKS[0])
for task in DEMO_TASKS:
    assert np.array_equal(first.render(),World(DEMO_SEED,task).render())
counterfactual=[]
fig,axes=plt.subplots(3,len(DEMO_TASKS),figsize=(12,8))
for j,task in enumerate(DEMO_TASKS):
    w=World(DEMO_SEED,task); actions,loc,maps=policy(w.render(),w.text,w.pusher)
    for i,(name,heat) in enumerate(zip(['colour','shape','joint'],maps)):
        axes[i,j].imshow(w.render(),extent=(0,1,1,0))
        axes[i,j].imshow(heat.reshape(GRID,GRID),extent=(0,1,1,0),alpha=.55,cmap='magma')
        axes[i,j].axis('off'); axes[i,j].set_title(f'{COLORS[task[0]]} {SHAPES[task[1]]} {task[2]:+d} / {name}',fontsize=9)
    result,frames=rollout(DEMO_SEED,task,record=True); counterfactual.append(result)
    print(w.text,':',result['reason'])
    annotated_gif(frames,f'counterfactual_{j}.gif','Same scene / changed instruction')
fig.tight_layout(); fig.savefig(RUN/'factorised_attention.png',dpi=160); plt.show()

## 9 · Your interactive policy playground

Choose an instruction, a scene seed, a perturbation and the number of actions executed between observations. Click **Run learned policy**. Reuse the same seed to compare instructions on an identical initial scene. This uses the trained neural policy, without retraining. The optional controls use standard `ipywidgets`; if it is unavailable, the preceding cell provides the same functionality through variables.

In [ ]:
def interactive_demo(color=0,shape=0,direction=1,seed=82001,condition='clean',execute=1):
    task=(int(color),int(shape),int(direction))
    result,frames=rollout(int(seed),task,condition,int(execute),record=True)
    print(instruction(*task),'|',result['reason'],'|',result['steps'],'steps |',result['model_calls'],'model calls')
    annotated_gif(frames,'interactive_demo.gif','Interactive policy rollout')
    return result

try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    color_control=widgets.Dropdown(options=list(zip(COLORS,range(3))),description='Colour')
    shape_control=widgets.Dropdown(options=list(zip(SHAPES,range(2))),description='Shape')
    direction_control=widgets.Dropdown(options=[('Left',-1),('Right',1)],value=1,description='Direction')
    seed_control=widgets.IntText(value=82001,description='Scene seed')
    condition_control=widgets.Dropdown(options=['clean','dark','noise','slow_actuator','displacement'],description='Test')
    chunk_control=widgets.Dropdown(options=[1,4],description='Execute')
    button=widgets.Button(description='Run learned policy',button_style='success',layout=widgets.Layout(width='200px'))
    demo_output=widgets.Output()
    def on_run(_):
        button.disabled=True
        try:
            with demo_output:
                clear_output(wait=True)
                interactive_demo(color_control.value,shape_control.value,direction_control.value,
                                 seed_control.value,condition_control.value,chunk_control.value)
        finally: button.disabled=False
    button.on_click(on_run)
    display(widgets.VBox([widgets.HBox([color_control,shape_control,direction_control]),
                          widgets.HBox([seed_control,condition_control,chunk_control]),button,demo_output]))
except ImportError:
    print('Optional widgets unavailable. Call interactive_demo(color=0, shape=1, direction=1) instead.')

## 10 · Export and write up the experiment

Keep the distinction between a successful demo and a general capability. Report seen and held-out tasks separately; include failures and expert ceilings. Executing longer chunks may reduce model calls without improving success. Camera and actuator tests explore different failure mechanisms.

**A useful next experiment:** set `LOCATION_WEIGHT=0` and `ATTRIBUTE_WEIGHT=0`, then rerun from scratch; or set `CAMERA_AUGMENTATION=False`. Compare on validation first and reserve fresh test seeds for the final comparison. To claim robustness improvement, train and evaluate a baseline without augmentation; the current benchmark alone only measures robustness levels.



In [ ]:
report={'version':2,'seed':SEED,'parameters':PARAMETERS,'device':str(DEVICE),'torch':torch.__version__,
        'train_episodes':TRAIN_EPISODES,'training_tasks':TRAIN_TASKS,'heldout_tasks':HELDOUT_TASKS,
        'steps_requested':STEPS,'steps_completed':step,'horizon':HORIZON,'batch_size':BATCH_SIZE,
        'learning_rate':LEARNING_RATE,'location_weight':LOCATION_WEIGHT,'attribute_weight':ATTRIBUTE_WEIGHT,
        'camera_augmentation':CAMERA_AUGMENTATION,
        'training_seconds':TRAIN_SECONDS,'evaluation_seconds':EVAL_SECONDS,
        'validation_before':baseline,'validation_after':after,'history':history,
        'benchmark':summary,'counterfactual':counterfactual,'development_rollouts':dev,
        'train_seeds':TRAIN_SEEDS,'val_seeds':VAL_SEEDS,'test_seeds':TEST_SEEDS,
        'heldout_seeds':HELDOUT_SEEDS,'stress_seeds':STRESS_SEEDS}
(RUN/'metrics.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
(RUN/'vocabulary.json').write_text(json.dumps(VOCAB,indent=2),encoding='utf-8')
with (RUN/'episodes.csv').open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
archive=Path('mini_vla_lab_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in RUN.iterdir():
        if p.is_file(): z.write(p,p.name)
print('Exported:',archive.resolve())
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    from IPython.display import FileLink
    display(FileLink(str(archive)))